# Semantic Segmentation with MobileNetV4

> Train a full semantic segmentation model using **MobileNetV4** as the backbone.

MobileNetV4 (ECCV 2024, arXiv:2404.10518) is Google's latest mobile-optimised ViT family.
Key architectural innovations:

| Innovation | Description |
|---|---|
| **UIB** (Universal Inverted Bottleneck) | Unifies IB, ConvNext, FFN, and ExtraDW blocks via NAS |
| **Mobile MQA** | Multi-Query Attention tuned for mobile accelerators (+39% speed) |
| **Hierarchical 4-stage design** | Multi-scale feature pyramid at strides 4, 8, 16, 32 |
| **Two families** | `conv` (conv-only) and `hybrid` (conv + attention) |

### Segmentation architecture

```
Image
  └─► MobileNetV4 backbone (features_only=True)
          │  P2 (stride 4)   ─────────────────────────┐
          │  P3 (stride 8)   ──────────────────────┐  │
          │  P4 (stride 16)  ───────────────────┐  │  │
          └  P5 (stride 32)  ─► FPN top-down ─► merge ─► merge ─► merge
                                                               ↓
                                                  segmentation head
                                                               ↓
                                                  bilinear ×4 upsample
                                                               ↓
                                              per-pixel class logits (H×W×C)
```

### Available MobileNetV4 checkpoints (timm)

| Model | Params | Speed | Notes |
|---|---|---|---|
| `mobilenetv4_conv_small.e2400_r224_in1k` | ~3.8 M | Fastest | Great baseline |
| `mobilenetv4_conv_medium.e250_r384_in12k_ft_in1k` | ~11 M | Fast | Better accuracy |
| `mobilenetv4_hybrid_medium.e500_r224_in1k` | ~11 M | Medium | Conv + attention |
| `mobilenetv4_conv_large.e600_r384_in1k` | ~32 M | Moderate | Best conv |
| `mobilenetv4_hybrid_large.e600_r384_in1k` | ~37 M | Moderate | Best overall |

### Requirements
- `timm >= 1.0.3`
- `torch >= 2.0`
- No gated weights — all checkpoints are publicly available

## 1. Installation

In [ ]:
!pip install -q --upgrade \
    'timm>=1.0.3' \
    'torch>=2.0' \
    torchvision \
    albumentations \
    matplotlib \
    scikit-learn \
    tqdm

## 2. Imports & Global Config

In [ ]:
import os
import random
import copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

print(f"PyTorch       : {torch.__version__}")
print(f"timm          : {timm.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device  : {device}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── Global configuration ──────────────────────────────────────────────────────
CFG = dict(
    # ---- Data ----------------------------------------------------------------
    # Expected layout (VOC-style):
    #   data_root/
    #       images/          <- RGB images  (.jpg / .png)
    #       masks/           <- single-channel PNG masks, pixel value = class id
    data_root       = "./seg_data",     # <-- change to your dataset path
    val_split       = 0.15,
    test_split      = 0.10,
    num_workers     = 4,
    ignore_index    = 255,              # mask pixel value to ignore in loss

    # ---- Classes -------------------------------------------------------------
    # List every class name in order (index 0 = background by convention)
    class_names     = ["background", "class_1", "class_2"],   # <-- edit

    # ---- Model ---------------------------------------------------------------
    # Pick any MobileNetV4 variant:
    #   mobilenetv4_conv_small.e2400_r224_in1k     (~3.8 M)  fastest
    #   mobilenetv4_conv_medium.e250_r384_in12k_ft_in1k (~11 M)
    #   mobilenetv4_hybrid_medium.e500_r224_in1k   (~11 M)  conv+attn
    #   mobilenetv4_conv_large.e600_r384_in1k      (~32 M)
    #   mobilenetv4_hybrid_large.e600_r384_in1k    (~37 M)  best accuracy
    backbone_name   = "mobilenetv4_conv_small.e2400_r224_in1k",
    pretrained      = True,
    # Feature pyramid stages: strides will be (4, 8, 16, 32)
    out_indices     = (1, 2, 3, 4),
    fpn_channels    = 128,              # FPN internal channel width
    freeze_backbone = False,            # set True for linear-probe style training

    # ---- Training ------------------------------------------------------------
    image_size      = 512,             # must be divisible by 32
    batch_size      = 8,
    epochs          = 40,
    lr              = 1e-3,
    backbone_lr     = 1e-4,            # lower LR for pretrained backbone
    weight_decay    = 1e-4,
    patience        = 8,
    dice_weight     = 0.5,             # weight of Dice loss (1-dice_weight = CE)
    output_dir      = "./mobilenetv4_seg",
)

CFG['num_classes'] = len(CFG['class_names'])
os.makedirs(CFG['output_dir'], exist_ok=True)
print("Config:", CFG)

## 3. Dataset

Expected folder layout:
```
seg_data/
├── images/
│   ├── 0001.jpg
│   └── ...
└── masks/
    ├── 0001.png   ← same stem, pixel value = class index
    └── ...
```
Mask pixel values must be integers in `[0, num_classes - 1]`.  
Use `ignore_index` (default 255) for unlabelled / void regions.

In [ ]:
class SegmentationDataset(Dataset):
    """Loads paired (image, mask) files from images/ and masks/ sub-folders."""

    IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

    def __init__(self, data_root: str, transform=None):
        self.img_dir  = Path(data_root) / 'images'
        self.msk_dir  = Path(data_root) / 'masks'
        self.transform = transform

        # Collect all image paths and resolve matching mask path
        self.samples = []
        for img_path in sorted(self.img_dir.iterdir()):
            if img_path.suffix.lower() not in self.IMG_EXTS:
                continue
            # Try .png mask first, then same extension
            msk_path = self.msk_dir / (img_path.stem + '.png')
            if not msk_path.exists():
                msk_path = self.msk_dir / img_path.name
            if msk_path.exists():
                self.samples.append((img_path, msk_path))

        assert len(self.samples) > 0, (
            f"No paired image/mask files found under {data_root}. "
            "Check that images/ and masks/ sub-folders exist."
        )
        print(f"Dataset: {len(self.samples)} pairs found in {data_root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, msk_path = self.samples[idx]
        image = np.array(Image.open(img_path).convert('RGB'))  # H W 3
        mask  = np.array(Image.open(msk_path))                 # H W  (uint8)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']   # (3, H, W) float tensor
            mask  = augmented['mask']    # (H, W)  long tensor

        return image, mask.long()

In [ ]:
# ── Albumentations transforms (handles image+mask jointly) ────────────────────
IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]
H = W    = CFG['image_size']

train_transform = A.Compose([
    A.RandomResizedCrop(height=H, width=W, scale=(0.5, 2.0), ratio=(0.75, 1.33)),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05, p=0.8),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(height=H, width=W),
    A.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ToTensorV2(),
])

In [ ]:
# ── Build dataset splits ──────────────────────────────────────────────────────
full_ds = SegmentationDataset(CFG['data_root'], transform=train_transform)

n_total = len(full_ds)
n_val   = int(n_total * CFG['val_split'])
n_test  = int(n_total * CFG['test_split'])
n_train = n_total - n_val - n_test

train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

# Apply val transforms to val / test splits (replace transform without re-reading files)
val_ds.dataset  = copy.copy(full_ds); val_ds.dataset.transform  = val_transform
test_ds.dataset = copy.copy(full_ds); test_ds.dataset.transform = val_transform

print(f"Train: {n_train} | Val: {n_val} | Test: {n_test}")

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)

In [ ]:
# ── Visualise samples ─────────────────────────────────────────────────────────
def denorm(t):
    t = t.clone()
    for c, m, s in zip(range(3), IMG_MEAN, IMG_STD):
        t[c] = t[c] * s + m
    return t.clamp(0, 1).permute(1, 2, 0).numpy()

def make_colormap(n):
    rng = np.random.RandomState(0)
    return rng.randint(0, 255, (n, 3), dtype=np.uint8)

COLORS = make_colormap(CFG['num_classes'])

def colorize_mask(mask_np):
    """Convert integer mask to an RGB image."""
    rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for cls_id, color in enumerate(COLORS):
        rgb[mask_np == cls_id] = color
    return rgb

imgs, masks = next(iter(train_loader))
n_show = min(4, len(imgs))
fig, axes = plt.subplots(n_show, 2, figsize=(10, 3 * n_show))
if n_show == 1: axes = axes[np.newaxis]
for i in range(n_show):
    axes[i, 0].imshow(denorm(imgs[i]))
    axes[i, 0].set_title('Image'); axes[i, 0].axis('off')
    axes[i, 1].imshow(colorize_mask(masks[i].numpy()))
    axes[i, 1].set_title('Mask');  axes[i, 1].axis('off')
patches = [mpatches.Patch(color=COLORS[c]/255, label=n)
           for c, n in enumerate(CFG['class_names'])]
fig.legend(handles=patches, loc='lower center', ncol=min(6, CFG['num_classes']), fontsize=8)
plt.suptitle('Sample training pairs', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Model: MobileNetV4 + FPN Decoder

### Architecture
```
MobileNetV4 backbone  →  4 feature maps at strides 4 / 8 / 16 / 32
                                ↓
FPN lateral 1×1 convs  →  project each to fpn_channels
                                ↓
Top-down path (add + 2× upsample)  →  single merged feature map at stride 4
                                ↓
Segmentation head (3×3 conv → 1×1 conv)
                                ↓
4× bilinear upsample   →  full-resolution logits (H × W × num_classes)
```

In [ ]:
# ── 4.1  Inspect backbone feature shapes ─────────────────────────────────────
_probe = timm.create_model(
    CFG['backbone_name'],
    features_only=True,
    pretrained=False,
    out_indices=CFG['out_indices'],
)
_probe.eval()
with torch.no_grad():
    _x   = torch.zeros(1, 3, CFG['image_size'], CFG['image_size'])
    _out = _probe(_x)

print(f"Backbone : {CFG['backbone_name']}")
print(f"Params   : {sum(p.numel() for p in _probe.parameters())/1e6:.1f}M")
print()
for i, f in zip(CFG['out_indices'], _out):
    stride = CFG['image_size'] // f.shape[-1]
    print(f"  stage {i} → stride {stride:>2}×  shape {tuple(f.shape)}")

BACKBONE_CHANNELS = [f.shape[1] for f in _out]   # save for model construction
del _probe, _out, _x

In [ ]:
# ── 4.2  FPN Decoder ─────────────────────────────────────────────────────────
class FPNDecoder(nn.Module):
    """Simple top-down Feature Pyramid Network decoder.

    Accepts a list of feature maps [P2, P3, P4, P5] (coarsest last)
    and returns a single fused feature map at the resolution of P2.
    """

    def __init__(self, in_channels: list[int], fpn_channels: int = 128):
        super().__init__()
        # Lateral 1×1 projections (one per scale)
        self.lateral = nn.ModuleList([
            nn.Conv2d(c, fpn_channels, kernel_size=1, bias=False)
            for c in in_channels
        ])
        # Smooth 3×3 conv applied after each top-down merge
        self.smooth = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(fpn_channels, fpn_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(fpn_channels),
                nn.ReLU(inplace=True),
            )
            for _ in in_channels
        ])

    def forward(self, features: list[torch.Tensor]) -> torch.Tensor:
        # features = [P2, P3, P4, P5]  (finest → coarsest)
        laterals = [lat(f) for lat, f in zip(self.lateral, features)]

        # Top-down path: start from coarsest (index -1)
        out = laterals[-1]
        for i in range(len(laterals) - 2, -1, -1):
            out = F.interpolate(out, size=laterals[i].shape[-2:],
                                mode='bilinear', align_corners=False)
            out = out + laterals[i]
            out = self.smooth[i](out)

        return out   # shape: (B, fpn_channels, H/4, W/4)

In [ ]:
# ── 4.3  Segmentation Head ───────────────────────────────────────────────────
class SegHead(nn.Module):
    """Light segmentation head: 3×3 conv → dropout → 1×1 classifier."""

    def __init__(self, in_channels: int, num_classes: int, dropout: float = 0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(in_channels, num_classes, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)

In [ ]:
# ── 4.4  Full Model ──────────────────────────────────────────────────────────
class MobileNetV4Seg(nn.Module):
    """MobileNetV4 backbone + FPN decoder + segmentation head.

    Output logits have the same spatial resolution as the input image.
    """

    def __init__(
        self,
        backbone_name : str,
        num_classes   : int,
        out_indices   : tuple = (1, 2, 3, 4),
        fpn_channels  : int   = 128,
        pretrained    : bool  = True,
        freeze_backbone: bool = False,
    ):
        super().__init__()

        # ── Backbone ──────────────────────────────────────────────────────────
        self.backbone = timm.create_model(
            backbone_name,
            features_only = True,
            pretrained    = pretrained,
            out_indices   = out_indices,
        )
        in_channels = self.backbone.feature_info.channels()

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # ── Decoder ───────────────────────────────────────────────────────────
        self.decoder = FPNDecoder(in_channels, fpn_channels)

        # ── Head ──────────────────────────────────────────────────────────────
        self.head = SegHead(fpn_channels, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        H, W = x.shape[-2:]

        features = self.backbone(x)       # list of (B, C_i, H_i, W_i)
        fused    = self.decoder(features) # (B, fpn_channels, H/4, W/4)
        logits   = self.head(fused)       # (B, num_classes, H/4, W/4)

        # Upsample to original resolution
        logits = F.interpolate(logits, size=(H, W),
                               mode='bilinear', align_corners=False)
        return logits                     # (B, num_classes, H, W)

In [ ]:
# ── Build & summarise model ───────────────────────────────────────────────────
model = MobileNetV4Seg(
    backbone_name  = CFG['backbone_name'],
    num_classes    = CFG['num_classes'],
    out_indices    = CFG['out_indices'],
    fpn_channels   = CFG['fpn_channels'],
    pretrained     = CFG['pretrained'],
    freeze_backbone= CFG['freeze_backbone'],
).to(device)

# Sanity check
with torch.no_grad():
    _dummy = torch.zeros(1, 3, CFG['image_size'], CFG['image_size']).to(device)
    _out   = model(_dummy)
print(f"Output shape : {tuple(_out.shape)}  (expected: 1 × {CFG['num_classes']} × {CFG['image_size']} × {CFG['image_size']})")
del _dummy, _out

total      = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
backbone_p = sum(p.numel() for p in model.backbone.parameters())
print(f"\nBackbone     : {backbone_p/1e6:.2f}M params")
print(f"Decoder+Head : {(total-backbone_p)/1e6:.2f}M params")
print(f"Trainable    : {trainable/1e6:.2f}M / {total/1e6:.2f}M")

## 5. Loss Functions & Metrics

We combine **Cross-Entropy** (per-pixel classification) with **Dice loss** (handles class imbalance well):  
`total_loss = (1 - dice_weight) × CE  +  dice_weight × Dice`

In [ ]:
class DiceLoss(nn.Module):
    """Soft multi-class Dice loss. Ignores pixels with label == ignore_index."""

    def __init__(self, num_classes: int, ignore_index: int = 255, smooth: float = 1.0):
        super().__init__()
        self.num_classes    = num_classes
        self.ignore_index   = ignore_index
        self.smooth         = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits:  (B, C, H, W)
        # targets: (B, H, W)  long
        probs = logits.softmax(dim=1)
        valid = (targets != self.ignore_index)              # (B, H, W) bool

        # One-hot encode valid targets only
        tgt_valid = targets.clone()
        tgt_valid[~valid] = 0
        one_hot = F.one_hot(tgt_valid, self.num_classes)    # (B, H, W, C)
        one_hot = one_hot.permute(0, 3, 1, 2).float()       # (B, C, H, W)

        # Zero out ignored positions
        mask = valid.unsqueeze(1).float()
        probs    = probs    * mask
        one_hot  = one_hot  * mask

        dims  = (0, 2, 3)
        inter = (probs * one_hot).sum(dims)
        union = probs.sum(dims) + one_hot.sum(dims)
        dice  = (2 * inter + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


class CombinedLoss(nn.Module):
    def __init__(self, num_classes, ignore_index=255, dice_weight=0.5):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.dice = DiceLoss(num_classes, ignore_index)
        self.w    = dice_weight

    def forward(self, logits, targets):
        return (1 - self.w) * self.ce(logits, targets) + self.w * self.dice(logits, targets)


criterion = CombinedLoss(
    num_classes   = CFG['num_classes'],
    ignore_index  = CFG['ignore_index'],
    dice_weight   = CFG['dice_weight'],
).to(device)
print("Loss: CrossEntropy + Dice  (weights:",
      f"{1-CFG['dice_weight']:.2f} / {CFG['dice_weight']:.2f})")

In [ ]:
# ── Metrics: pixel accuracy + mean IoU ───────────────────────────────────────
class SegMetrics:
    """Accumulates predictions over an epoch and computes mIoU + pixel acc."""

    def __init__(self, num_classes: int, ignore_index: int = 255):
        self.C            = num_classes
        self.ignore_index = ignore_index
        self.reset()

    def reset(self):
        self.confusion = np.zeros((self.C, self.C), dtype=np.int64)

    def update(self, preds: torch.Tensor, targets: torch.Tensor):
        """preds: (B, H, W) long  |  targets: (B, H, W) long"""
        p = preds.cpu().numpy().ravel()
        t = targets.cpu().numpy().ravel()
        valid = t != self.ignore_index
        p, t  = p[valid], t[valid]
        np.add.at(self.confusion, (t, p), 1)

    def compute(self) -> dict:
        cm      = self.confusion
        diag    = np.diag(cm)
        row_sum = cm.sum(axis=1)
        col_sum = cm.sum(axis=0)
        iou     = diag / np.maximum(row_sum + col_sum - diag, 1)
        miou    = np.nanmean(iou[row_sum > 0])
        pix_acc = diag.sum() / np.maximum(cm.sum(), 1)
        return {'miou': float(miou), 'pixel_acc': float(pix_acc),
                'iou_per_class': iou.tolist()}

## 6. Optimizer & Scheduler

In [ ]:
# Two parameter groups: lower LR for pretrained backbone, higher for decoder+head
backbone_params = list(model.backbone.parameters())
decoder_params  = list(model.decoder.parameters()) + list(model.head.parameters())

optimizer = optim.AdamW(
    [
        {'params': [p for p in backbone_params if p.requires_grad],
         'lr'    : CFG['backbone_lr']},
        {'params': decoder_params,
         'lr'    : CFG['lr']},
    ],
    weight_decay = CFG['weight_decay'],
)

warmup_epochs = max(1, CFG['epochs'] // 10)
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs),
        optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                              T_max=CFG['epochs'] - warmup_epochs,
                                              eta_min=1e-7),
    ],
    milestones=[warmup_epochs],
)

print(f"Backbone LR : {CFG['backbone_lr']}")
print(f"Decoder LR  : {CFG['lr']}")
print(f"Warm-up     : {warmup_epochs} epochs → CosineAnnealing")

## 7. Training Loop

In [ ]:
def run_epoch(model, loader, criterion, metrics, optimizer=None, phase='train'):
    """One epoch of training or validation.
    Returns (avg_loss, miou, pixel_acc).
    """
    is_train = (phase == 'train')
    model.train(is_train)
    metrics.reset()
    total_loss = 0.0

    with torch.set_grad_enabled(is_train):
        for imgs, masks in tqdm(loader, desc=phase, leave=False):
            imgs  = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            logits = model(imgs)                        # (B, C, H, W)
            loss   = criterion(logits, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

            preds = logits.argmax(dim=1)                # (B, H, W)
            metrics.update(preds, masks)
            total_loss += loss.item() * imgs.size(0)

    stats = metrics.compute()
    return total_loss / len(loader.dataset), stats['miou'], stats['pixel_acc']

In [ ]:
train_metrics = SegMetrics(CFG['num_classes'], CFG['ignore_index'])
val_metrics   = SegMetrics(CFG['num_classes'], CFG['ignore_index'])

history = {k: [] for k in ['train_loss','val_loss','train_miou','val_miou',
                             'train_pix_acc','val_pix_acc']}
best_miou  = 0.0
no_improve = 0
best_ckpt  = os.path.join(CFG['output_dir'], 'best_model.pt')

print(f"\nTraining MobileNetV4-Seg for {CFG['epochs']} epochs ...\n")

for epoch in range(1, CFG['epochs'] + 1):
    tr_loss, tr_miou, tr_acc = run_epoch(
        model, train_loader, criterion, train_metrics, optimizer, 'train')
    vl_loss, vl_miou, vl_acc = run_epoch(
        model, val_loader, criterion, val_metrics, None, 'val')
    scheduler.step()

    for k, v in zip(history.keys(),
                    [tr_loss, vl_loss, tr_miou, vl_miou, tr_acc, vl_acc]):
        history[k].append(v)

    lr_now = optimizer.param_groups[-1]['lr']
    print(f"Ep {epoch:>3}/{CFG['epochs']} | LR {lr_now:.2e}"
          f" | Train loss {tr_loss:.4f}  mIoU {tr_miou:.4f}  acc {tr_acc:.4f}"
          f" | Val   loss {vl_loss:.4f}  mIoU {vl_miou:.4f}  acc {vl_acc:.4f}",
          end='')

    if vl_miou > best_miou:
        best_miou  = vl_miou
        no_improve = 0
        torch.save(model.state_dict(), best_ckpt)
        print('  ← best')
    else:
        no_improve += 1
        print(f'  (no improve {no_improve}/{CFG["patience"]})')

    if no_improve >= CFG['patience']:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nBest val mIoU : {best_miou:.4f}')
print(f'Checkpoint    : {best_ckpt}')

## 8. Training Curves

In [ ]:
ep = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(ep, history['train_loss'], label='Train')
axes[0].plot(ep, history['val_loss'],   label='Val')
axes[0].set_title('Combined Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(ep, history['train_miou'], label='Train')
axes[1].plot(ep, history['val_miou'],   label='Val')
axes[1].set_title('mIoU'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(ep, history['train_pix_acc'], label='Train')
axes[2].plot(ep, history['val_pix_acc'],   label='Val')
axes[2].set_title('Pixel Accuracy'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.suptitle(
    f'MobileNetV4-Seg — {CFG["backbone_name"].split(".")[0]}',
    fontsize=12
)
plt.tight_layout()
plt.savefig(os.path.join(CFG['output_dir'], 'training_curves.png'), dpi=150)
plt.show()

## 9. Evaluation on Test Set

In [ ]:
# Reload best checkpoint
model.load_state_dict(torch.load(best_ckpt, map_location=device))
model.eval()

test_metrics = SegMetrics(CFG['num_classes'], CFG['ignore_index'])
test_loss    = 0.0

with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc='Test set'):
        imgs  = imgs.to(device)
        masks = masks.to(device)
        logits = model(imgs)
        test_loss += criterion(logits, masks).item() * imgs.size(0)
        test_metrics.update(logits.argmax(dim=1), masks)

test_loss /= len(test_loader.dataset)
stats = test_metrics.compute()

print(f"\nTest Loss      : {test_loss:.4f}")
print(f"Test mIoU      : {stats['miou']:.4f}")
print(f"Test Pixel Acc : {stats['pixel_acc']:.4f}")
print("\nPer-class IoU:")
for cls_id, (name, iou) in enumerate(zip(CFG['class_names'], stats['iou_per_class'])):
    print(f"  [{cls_id}] {name:<20s}: {iou:.4f}")

In [ ]:
# Per-class IoU bar chart
ious = stats['iou_per_class']
fig, ax = plt.subplots(figsize=(max(8, CFG['num_classes'] * 0.7), 4))
bars = ax.bar(CFG['class_names'], ious,
              color=[c/255 for c in COLORS[:CFG['num_classes']]])
ax.axhline(stats['miou'], color='red', linestyle='--', label=f"mIoU={stats['miou']:.3f}")
ax.set_ylim(0, 1)
ax.set_ylabel('IoU')
ax.set_title('Per-class IoU on Test Set')
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(CFG['output_dir'], 'per_class_iou.png'), dpi=150)
plt.show()

## 10. Qualitative Visualisation

In [ ]:
def overlay_mask(image_np, mask_np, alpha=0.5):
    """Blend RGB image with colorised mask."""
    color = colorize_mask(mask_np).astype(float)
    return np.clip(image_np * (1 - alpha) + color * alpha / 255, 0, 1)


model.eval()
imgs, masks = next(iter(test_loader))
with torch.no_grad():
    preds = model(imgs.to(device)).argmax(dim=1).cpu()

n_show = min(4, len(imgs))
fig, axes = plt.subplots(n_show, 3, figsize=(14, 4 * n_show))
if n_show == 1: axes = axes[np.newaxis]

for i in range(n_show):
    img_np  = denorm(imgs[i])                          # H W 3  float [0,1]
    gt_np   = masks[i].numpy()
    pred_np = preds[i].numpy()

    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title('Image'); axes[i, 0].axis('off')

    axes[i, 1].imshow(overlay_mask(img_np, gt_np))
    axes[i, 1].set_title('Ground Truth'); axes[i, 1].axis('off')

    axes[i, 2].imshow(overlay_mask(img_np, pred_np))
    axes[i, 2].set_title('Prediction'); axes[i, 2].axis('off')

patches = [mpatches.Patch(color=COLORS[c]/255, label=n)
           for c, n in enumerate(CFG['class_names'])]
fig.legend(handles=patches, loc='lower center', ncol=min(6, CFG['num_classes']), fontsize=8)
plt.suptitle('Qualitative results on test set', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(CFG['output_dir'], 'qualitative.png'), dpi=150)
plt.show()

## 11. Inference on a Single Image

In [ ]:
@torch.no_grad()
def segment_image(image_path: str, model: nn.Module) -> np.ndarray:
    """Return predicted mask (H, W) for a single image file."""
    img  = np.array(Image.open(image_path).convert('RGB'))
    orig_h, orig_w = img.shape[:2]

    tensor = val_transform(image=img)['image'].unsqueeze(0).to(device)
    logits = model(tensor)                              # (1, C, H, W)
    pred   = logits.argmax(dim=1)[0].cpu().numpy()     # (H, W)

    # Resize back to original resolution
    pred_full = np.array(
        Image.fromarray(pred.astype(np.uint8)).resize(
            (orig_w, orig_h), Image.NEAREST
        )
    )

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img);                        axes[0].set_title('Input');       axes[0].axis('off')
    axes[1].imshow(colorize_mask(pred_full));   axes[1].set_title('Pred mask');   axes[1].axis('off')
    axes[2].imshow(overlay_mask(img.astype(float)/255, pred_full))
    axes[2].set_title('Overlay');  axes[2].axis('off')

    patches = [mpatches.Patch(color=COLORS[c]/255, label=n)
               for c, n in enumerate(CFG['class_names'])]
    fig.legend(handles=patches, loc='lower center', ncol=min(6, CFG['num_classes']), fontsize=8)
    plt.tight_layout()
    plt.show()
    return pred_full


# Example — replace with your image path:
# pred_mask = segment_image('./my_image.jpg', model)
print("segment_image() helper defined. Uncomment the call above to run.")

## 12. Save & Export

In [ ]:
# ── Full checkpoint ───────────────────────────────────────────────────────────
final_ckpt = os.path.join(CFG['output_dir'], 'mobilenetv4_seg_final.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names'     : CFG['class_names'],
    'num_classes'     : CFG['num_classes'],
    'backbone_name'   : CFG['backbone_name'],
    'out_indices'     : CFG['out_indices'],
    'fpn_channels'    : CFG['fpn_channels'],
    'best_val_miou'   : best_miou,
    'image_size'      : CFG['image_size'],
}, final_ckpt)
print(f"Saved: {final_ckpt}")

In [ ]:
# ── Reload helper ─────────────────────────────────────────────────────────────
def load_mobilenetv4_seg(checkpoint_path: str):
    """Reload a saved MobileNetV4Seg model."""
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    m = MobileNetV4Seg(
        backbone_name = ckpt['backbone_name'],
        num_classes   = ckpt['num_classes'],
        out_indices   = tuple(ckpt['out_indices']),
        fpn_channels  = ckpt['fpn_channels'],
        pretrained    = False,
    )
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    print(f"Loaded | classes: {ckpt['class_names']} | val mIoU: {ckpt['best_val_miou']:.4f}")
    return m, ckpt['class_names']


# loaded_model, loaded_classes = load_mobilenetv4_seg(final_ckpt)
print("load_mobilenetv4_seg() defined. Uncomment above to test reloading.")

In [ ]:
# ── (Optional) Export to ONNX ─────────────────────────────────────────────────
# Useful for deployment on mobile devices or inference servers.

def export_onnx(model, image_size, output_path):
    model.eval()
    dummy = torch.zeros(1, 3, image_size, image_size)
    torch.onnx.export(
        model, dummy, output_path,
        input_names  = ['image'],
        output_names = ['logits'],
        dynamic_axes = {'image': {0: 'batch'}, 'logits': {0: 'batch'}},
        opset_version = 17,
    )
    print(f"ONNX model saved: {output_path}")

# onnx_path = os.path.join(CFG['output_dir'], 'mobilenetv4_seg.onnx')
# export_onnx(model.cpu(), CFG['image_size'], onnx_path)
print("export_onnx() defined. Uncomment above to export.")

## Summary

### Architecture recap

```
MobileNetV4 backbone (timm, features_only=True)
    ├── stage 1 → stride  4  ┐
    ├── stage 2 → stride  8  │
    ├── stage 3 → stride 16  │  FPN top-down decoder
    └── stage 4 → stride 32  ┘      ↓
                               SegHead (3×3 → 1×1)
                                       ↓
                               4× bilinear upsample
                                       ↓
                            logits  (B, C, H, W)
```

### Key design choices

| Choice | Rationale |
|---|---|
| FPN decoder | Fuses multi-scale features; simple yet effective |
| CE + Dice loss | CE ensures per-pixel accuracy; Dice handles class imbalance |
| Differential LR | Lower LR for pretrained backbone prevents catastrophic forgetting |
| Albumentations | Handles geometric transforms consistently on image+mask pairs |
| mIoU early-stopping | More meaningful than loss for segmentation tasks |

### Backbone trade-offs

| Backbone | Speed | Accuracy | Best for |
|---|---|---|---|
| `conv_small` | ★★★★★ | ★★★ | Mobile / edge deployment |
| `conv_medium` | ★★★★ | ★★★★ | Balanced production use |
| `hybrid_medium` | ★★★ | ★★★★ | When accuracy > speed |
| `hybrid_large` | ★★ | ★★★★★ | Research / server inference |

### Tips
- **Small dataset (< 2 k images)**: set `freeze_backbone=True` and train decoder only first, then unfreeze.
- **Resolution**: larger input (640, 768) improves small-object IoU at the cost of memory.
- **Class imbalance**: increase `dice_weight` toward 0.7–0.8 for highly imbalanced datasets.
- **Multi-scale inference**: average logits over flipped/scaled inputs at test time for +1–2 mIoU.

### References
- [MobileNetV4 paper (arXiv:2404.10518)](https://arxiv.org/abs/2404.10518)
- [timm MobileNetV4 collection](https://huggingface.co/collections/timm/mobilenetv4-pretrained-weights-6669c22cda4db4244def9637)
- [timm features_only docs](https://huggingface.co/docs/timm/feature_extraction)